# Day 04 프로젝트 실습
## SQLite로 RDB 구축하고 데이터 저장하기

> 이 노트북은 **[DBMS 개요](../1.%20DBMS%20개요.pdf)** · **[데이터 모델링](../2.%20데이터%20모델링.pdf)** · **[데이터베이스 정규화](../3.%20데이터베이스%20정규화.pdf)** 에서 배운 개념을,
> **[Day03 데이터 정의서.xlsx](../../day03_3일차/프로젝트/데이터%20정의서.xlsx)** 에서 정의한 3개 데이터(상품, 영양소_기능성, 영양소_섭취기준)에 그대로 적용하는 실습입니다.

### 실습 목표
1. 데이터 정의서를 기준으로 **테이블 스키마**(컬럼, 자료형, 기본키)를 설계한다.
2. `sqlite3`로 실제 데이터베이스 파일을 만들고 테이블을 생성한다.
3. `data/` 폴더의 CSV 데이터를 테이블에 저장(INSERT)한다.
4. SQL로 데이터를 조회(SELECT)해 원하는 정보를 꺼내본다.

### 사용하는 데이터 (`data/` 폴더)
| 파일 | Day03 데이터 정의서 시트 | 비고 |
| --- | --- | --- |
| `상품마스터.csv` | 상품_데이터 | 상품ID가 자연키(고유값) |
| `영양소_기능성_추천DB_확장판.csv` | 영양소_기능성_데이터 | 고유키가 없는 사실(fact) 테이블 |
| `영양소별_권장섭취량_확장판_한글.csv` | 영양소_섭취기준_데이터 | 고유키가 없는 사실(fact) 테이블 |


---
## 0. 준비 : 라이브러리와 데이터 불러오기

`pyproject.toml`에 정의된 라이브러리를 사용합니다.

```
jupyter
pandas
```

`sqlite3`는 파이썬 표준 라이브러리라서 별도 설치 없이 바로 사용할 수 있습니다.


In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


In [2]:
product = pd.read_csv("data/상품마스터.csv")
func_db = pd.read_csv("data/영양소_기능성_추천DB_확장판.csv")
rda_db = pd.read_csv("data/영양소별_권장섭취량_확장판_한글.csv")

print("상품마스터:", product.shape)
print("영양소_기능성_추천DB:", func_db.shape)
print("영양소별_권장섭취량:", rda_db.shape)


상품마스터: (44, 15)
영양소_기능성_추천DB: (54, 10)
영양소별_권장섭취량: (228, 13)


---
## 1. 테이블 스키마 설계 (정규화 관점)

Day03 데이터 정의서의 `데이터_식별` 시트에서 정리한 3개 데이터 묶음을 각각 하나의 테이블로 설계합니다.

| 테이블명 | 기본키(PK) | 비고 |
| --- | --- | --- |
| `상품` | `상품ID` (자연키) | CSV에 이미 고유한 상품ID가 있어 그대로 기본키로 사용 |
| `영양소_기능성` | `id` (대체키, AUTOINCREMENT) | 같은 영양소가 여러 행에 반복돼 자연키가 없으므로 대체키 사용 |
| `영양소_섭취기준` | `id` (대체키, AUTOINCREMENT) | 같은 영양소가 연령대·성별별로 반복돼 자연키가 없으므로 대체키 사용 |

세 테이블은 **정식 외래키(FK) 제약조건 대신, 값이 같은 컬럼끼리 매칭하는 방식**으로 연결됩니다. (관련 개념: 후보키·매칭키, `3. 데이터베이스 정규화.pdf`)

- `상품.검색키워드(주요원료)` ↔ `영양소_기능성.영양소`
- `상품.관련_증상` ↔ `영양소_기능성.관련_증상`
- `상품.검색키워드(주요원료)` ↔ `영양소_섭취기준.영양소`

> `상한섭취량` 컬럼에는 `350(마그네슘 예시)`처럼 숫자에 설명이 덧붙은 값이 섞여 있어 숫자(REAL)가 아닌 문자(TEXT)로 저장합니다. (Day02 데이터 품질 점검과 연결되는 부분입니다.)


In [3]:
DB_PATH = Path("건강한하루.db")

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()


In [4]:
cur.executescript("""
DROP TABLE IF EXISTS 상품;
DROP TABLE IF EXISTS 영양소_기능성;
DROP TABLE IF EXISTS 영양소_섭취기준;

CREATE TABLE 상품 (
    상품ID TEXT PRIMARY KEY,
    상품명 TEXT NOT NULL,
    업소명 TEXT NOT NULL,
    카테고리 TEXT NOT NULL,
    "검색키워드(주요원료)" TEXT NOT NULL,
    관련_증상 TEXT NOT NULL,
    신고번호 TEXT,
    등록일자 TEXT,
    소비기한 TEXT,
    성상 TEXT,
    섭취량_섭취방법 TEXT,
    섭취시주의사항 TEXT,
    기능성_내용 TEXT,
    신고번호_상세조회URL TEXT,
    데이터출처 TEXT
);

CREATE TABLE 영양소_기능성 (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    영양소 TEXT NOT NULL,
    분류 TEXT,
    기능성 TEXT NOT NULL,
    관련_증상 TEXT,
    키워드 TEXT,
    근거_수준 TEXT,
    근거_설명 TEXT,
    신뢰도 TEXT,
    출처 TEXT,
    출처_URL TEXT
);

CREATE TABLE 영양소_섭취기준 (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    영양소 TEXT NOT NULL,
    연령대 TEXT,
    성별 TEXT,
    임신여부 TEXT,
    수유여부 TEXT,
    권장섭취량 TEXT,
    충분섭취량 TEXT,
    상한섭취량 TEXT,
    단위 TEXT,
    출처 TEXT,
    기관 TEXT,
    비고 TEXT,
    출처_URL TEXT
);
""")

conn.commit()
print("테이블 생성 완료")


테이블 생성 완료


---
## 2. 데이터 저장 (INSERT)

`DataFrame.to_sql(..., if_exists="append")`을 사용하면, 위에서 만든 테이블 구조(스키마)에 맞춰 CSV의 각 행을 그대로 저장할 수 있습니다.

`신고번호`는 CSV에서 숫자로 읽히지만 테이블에는 TEXT로 정의했으므로, 저장 전에 문자열로 변환합니다.


In [5]:
product["신고번호"] = product["신고번호"].astype(str)

product.to_sql("상품", conn, if_exists="append", index=False)
func_db.to_sql("영양소_기능성", conn, if_exists="append", index=False)
rda_db.to_sql("영양소_섭취기준", conn, if_exists="append", index=False)

conn.commit()

for table in ["상품", "영양소_기능성", "영양소_섭취기준"]:
    count = cur.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table} : {count}행 저장됨")


상품 : 44행 저장됨
영양소_기능성 : 54행 저장됨
영양소_섭취기준 : 228행 저장됨


---
## 3. 데이터 조회 (SELECT) 예시

`sqlite3`로 직접 `cursor.execute()`를 실행할 수도 있고, `pandas.read_sql_query()`로 조회 결과를 바로 DataFrame으로 받을 수도 있습니다. 실습에서는 결과를 표로 확인하기 편한 `pandas.read_sql_query()`를 주로 사용합니다.


### 3-1. 기본 조회 : 상품 테이블 전체보기 (상위 5건)

In [6]:
pd.read_sql_query("SELECT * FROM 상품 LIMIT 5", conn)


,상품ID,상품명,업소명,카테고리,검색키워드(주요원료),관련_증상,신고번호,등록일자,소비기한,성상,섭취량_섭취방법,섭취시주의사항,기능성_내용,신고번호_상세조회URL,데이터출처
0,P001,뉴메릿 비타민C&D 메가,(주)푸드어셈블,비타민,비타민 C,피로,2025001237116,2026-07-08,제조일로부터 24개월,"고유의 향미가 있고 이미, 이취가 없는 하양색의 입자성이 없는 분말","1일1회 , 1회 1포(2.1g)를 물과 함께 섭취하십시오",1. 고칼슘혈증이 있거나 의약품 복용 시 전문가와 상담할 것 / 2. 이상사례 발생...,[비타민C] / (1) 결합조직 형성과 기능유지에 필요 / (2) 철의 흡수에 필요...,https://www.foodsafetykorea.go.kr/portal/healt...,식품안전나라(foodsafetykorea.go.kr) 건강기능식품 검색
1,P002,뉴메릿 비타민C&D 듀얼 메가,(주)푸드어셈블,비타민,비타민 C,피로,2025001237117,2026-07-08,제조일로부터 24개월,"고유의 향미가 있고 이미, 이취가 없는 하양색 분말","1일 1회 , 1회 1포(3.2g)를 물과 함께 섭취하십시오.",➀ 고칼슘혈증이 있거나 의약품 복용 시 전문가와 상담할 것 / ➁ 이상사례 발생 시...,[비타민C] / (1) 결합조직 형성과 기능유지에 필요 / (2) 철의 흡수에 필요...,https://www.foodsafetykorea.go.kr/portal/healt...,식품안전나라(foodsafetykorea.go.kr) 건강기능식품 검색
2,P003,마이니 뼈건강 칼슘 마그네슘 비타민D,주식회사 노바렉스,비타민,비타민 D,골 건강,200400200083446,2026-07-09,제조일로부터 24개월,고유의 향미가 있고 이미·이취가 없는 점박이를 포함한 흰색의 흰색 장방형 제피 정제,"1일 1회, 1회 1정을 물과 함께 섭취하십시오.","특정질환, 특이체질, 알레르기체질, 임산부의 경우에는 간혹 개인에 따라 과민반응이 ...","[칼슘] 뼈와 치아 형성에 필요, 신경과 근육 기능 유지에 필요, 정상적인 혈액응고...",https://www.foodsafetykorea.go.kr/portal/healt...,식품안전나라(foodsafetykorea.go.kr) 건강기능식품 검색
3,P004,이지맘 비타민D,(주)비오팜2공장,비타민,비타민 D,골 건강,2024002023371,2026-07-08,제조일로부터 24개월,"고유의 향미가 있고 이미, 이취가 없는 연한 노랑색의 내용물을 함유한 투명한 타원형...","1일 1회, 1회 1캡슐을 물과 함께 섭취하십시오.",- 고칼슘혈증이 있거나 의약품 복용 시 전문가와 상담할 것 / - 이상사례 발생 시...,[비타민D] (가) 칼슘과 인이 흡수되고 이용되는데 필요 (나) 뼈의 형성과 유지에...,https://www.foodsafetykorea.go.kr/portal/healt...,식품안전나라(foodsafetykorea.go.kr) 건강기능식품 검색
4,P005,오메가 비타민E 미니,코스맥스바이오(주),비타민,비타민 E,노화,200400200024068,2026-05-08,제조일로부터 2년,"고유의 향미가 있고 이미, 이취가 없는 연한 노란색의 내용물을 함유한 투명한 타원형...","1일 2회, 1회 5캡슐을 충분한 물과 함께 섭취하십시오. 1일 1회, 1회 10캡...",섭취 시 목에 걸릴 수 있으므로 반드시 물과 함께 섭취하십시오. / 섭취 시 위장장...,1) EPA 및 DHA 함유 유지 : 혈중 중성지질 개선·혈행 개선·건조한 눈을 개...,https://www.foodsafetykorea.go.kr/portal/healt...,식품안전나라(foodsafetykorea.go.kr) 건강기능식품 검색


### 3-2. 조건 조회 : 카테고리가 '비타민'인 상품만 보기

In [7]:
query = """
SELECT 상품ID, 상품명, 카테고리, "검색키워드(주요원료)" AS 주요원료
FROM 상품
WHERE 카테고리 = ?
"""

pd.read_sql_query(query, conn, params=("비타민",))


,상품ID,상품명,카테고리,주요원료
0,P001,뉴메릿 비타민C&D 메가,비타민,비타민 C
1,P002,뉴메릿 비타민C&D 듀얼 메가,비타민,비타민 C
2,P003,마이니 뼈건강 칼슘 마그네슘 비타민D,비타민,비타민 D
3,P004,이지맘 비타민D,비타민,비타민 D
4,P005,오메가 비타민E 미니,비타민,비타민 E
5,P006,백수원 비타민E 프리미엄,비타민,비타민 E
6,P031,임신은 처음이라 활성형 엽산 800,비타민,엽산
7,P032,활성엽산 800+ 미니,비타민,엽산
8,P033,프라임 비오틴 10000,비타민,비오틴
9,P034,데일리 비오틴10000,비타민,비오틴


### 3-3. 집계 조회 : 카테고리별 상품 수

In [8]:
query = """
SELECT 카테고리, COUNT(*) AS 상품수
FROM 상품
GROUP BY 카테고리
ORDER BY 상품수 DESC
"""

pd.read_sql_query(query, conn)


,카테고리,상품수
0,기능성 원료,26
1,비타민,10
2,무기질,8


### 3-4. JOIN 조회 : 상품의 추천 근거(기능성) 함께 보기

`상품.검색키워드(주요원료)` = `영양소_기능성.영양소` 이면서 `상품.관련_증상` = `영양소_기능성.관련_증상` 인 행을 연결해, RF-1.3(추천 근거 표시)에 필요한 형태로 조회합니다.


In [9]:
query = """
SELECT
    p.상품ID,
    p.상품명,
    p."검색키워드(주요원료)" AS 주요원료,
    f.기능성,
    f.근거_설명,
    f.신뢰도
FROM 상품 AS p
JOIN 영양소_기능성 AS f
    ON p."검색키워드(주요원료)" = f.영양소
   AND p.관련_증상 = f.관련_증상
WHERE p.상품ID = ?
"""

pd.read_sql_query(query, conn, params=("P001",))


,상품ID,상품명,주요원료,기능성,근거_설명,신뢰도
0,P001,뉴메릿 비타민C&D 메가,비타민 C,항산화,식품의약품안전처에서 인정한 건강기능식품 기능성 근거,매우 높음


### 3-5. JOIN 조회 : 상품 + 섭취기준 함께 보기

특정 연령대·성별 사용자에게 상품의 권장/상한 섭취량을 함께 보여주는, RF-4.1(권장/상한 섭취량 표시)에 필요한 형태입니다.


In [10]:
query = """
SELECT
    p.상품명,
    p."검색키워드(주요원료)" AS 주요원료,
    s.연령대,
    s.성별,
    s.권장섭취량,
    s.상한섭취량,
    s.단위
FROM 상품 AS p
JOIN 영양소_섭취기준 AS s
    ON p."검색키워드(주요원료)" = s.영양소
WHERE p.상품ID = ?
  AND s.연령대 = ?
  AND s.성별 = ?
"""

pd.read_sql_query(query, conn, params=("P001", "19~29세", "여성"))


,상품명,주요원료,연령대,성별,권장섭취량,상한섭취량,단위
0,뉴메릿 비타민C&D 메가,비타민 C,19~29세,여성,100.0,2000,mg


---
## 정리

- `데이터 정의서`(Day03)의 데이터/항목 정의를 그대로 `CREATE TABLE`의 컬럼과 자료형으로 옮겼습니다.
- 자연키가 있으면 그대로 기본키로 쓰고(`상품ID`), 자연키가 없는 사실(fact) 데이터는 대체키(`id AUTOINCREMENT`)를 사용했습니다.
- CSV 데이터를 `to_sql()`로 테이블에 저장하고, `SELECT`·`WHERE`·`GROUP BY`·`JOIN`으로 필요한 정보를 조회했습니다.
- 다음 시간에는 이 SQLite 데이터베이스를 실제 서비스 기능(추천, 복용 안내 등)과 연결하는 방법을 다룹니다.


In [11]:
conn.close()
